In [1]:
import numpy as np
import sys
import os
from autokmc.structure import build_surface, build_nanoparticle
from ase.visualize.x3d import view_x3d
from autokmc.surface import find_surface_atoms
import copy
from collections import Counter
from autokmc.graph import build_graph
from autokmc.site import find_adsorption_sites

In [2]:
# Ensure the project root is on the path when running from the autokmc/ subdirectory
sys.path.insert(0, os.path.abspath("../.."))

In [3]:
### Load the Allegro/NequIP calculator
import torch
from nequip.ase import NequIPCalculator

_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
_MODEL_FILE = "asehcocuau.nequip.pt2" if _DEVICE == "cuda" else "cpuhcocuau.nequip.pth"
_MODEL_PATH = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", _MODEL_FILE)
print(f"Using device : {_DEVICE}")
print(f"Model file   : {_MODEL_FILE}")

def make_calc():
    """Return a fresh NequIPCalculator instance loaded from the model."""
    return NequIPCalculator.from_compiled_model(
        compile_path=_MODEL_PATH,
        device=_DEVICE,
    )

calc = make_calc()
print(f"Calculator : {calc.__class__.__name__}")
print(f"Model      : {_MODEL_PATH}")

Using device : cpu
Model file   : cpuhcocuau.nequip.pth
Calculator : NequIPCalculator
Model      : ./cpuhcocuau.nequip.pth


/Users/bunting4/local/lib/python3.13/site-packages/nequip/ase/nequip_calculator.py:137: UserWarning: Trying to use model type names as chemical symbols; this may not be correct for your model (and may cause an error if model type names are not chemical symbols)! To avoid this warning, please provide `chemical_symbols` explicitly.
  warnings.warn(


In [4]:
## Build a Cu(111) surface slab
slab = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(1, 1, 1),
    calculator=make_calc(),
    min_slab_size=8.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True
)

print(f"\nSlab formula : {slab.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab)}")
cell = slab.get_cell()
print(f"Cell (Å)     : a={cell[0,0]:.3f}  b={cell[1,1]:.3f}  c={cell[2,2]:.3f}")

  Bulk Cu (FCC) optimised
  Converged : True  |  steps : 2
  E/atom    : -10.72548 eV
  Opt a     : 3.5922 Å

Building Cu(111) slab [FCC]  (pymatgen SlabGenerator) ...
  Orthogonal transform : (1,-1,1,1)  det=2
  Cell after ortho     : a=2.540  b=4.399  c=20.739 Å
  Tiling 5×3 → 120 atoms
  Surface Cu(111)  [FCC]
  Formula        : Cu120
  Atoms          : 120
  Cu             : 120     (100.0 %)
  Cell (Å)       : a=12.700  b=13.198  c=20.739
  z range        : [7.26, 13.48] Å
  Fixing bottom 2 layer(s): 60 atoms
  Optimisation : converged=True steps=4  E=-1260.7390 eV  E/atom=-10.5062 eV/atom

Slab formula : Cu120
Slab atoms   : 120
Cell (Å)     : a=12.700  b=13.198  c=20.739


In [5]:
## Visualise the slab
view_x3d(slab)

In [6]:
### Get surface atoms

surface_mask, surface_indices, method = find_surface_atoms(
    slab,
    which="top",
    tag_atoms=True,   # writes slab.arrays["surface"] for extxyz export
)

print(f"Detection method : {method}")
print(f"Surface atoms    : {surface_mask.sum()} / {len(slab)}")
print(f"Surface indices  : {surface_indices}")

Detection method : raycasting
Surface atoms    : 30 / 120
Surface indices  : [  3   7  11  15  19  23  27  31  35  39  43  47  51  55  59  63  67  71
  75  79  83  87  91  95  99 103 107 111 115 119]


In [7]:
### Visualise surface atoms in X3D
# Surface atoms are shown as Au (gold), bulk atoms remain as Cu
# so the two populations are visually distinct in x3d.
slab_vis = copy.deepcopy(slab)
symbols = np.array(slab_vis.get_chemical_symbols())
symbols[surface_indices] = "Au"
slab_vis.set_chemical_symbols(symbols.tolist())

view_x3d(slab_vis)

In [8]:
### Build the graph for the slab
graph = build_graph(slab)
print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
# Show breakdown by node type
type_counts = Counter(d["type"] for _, d in graph.nodes(data=True))
for t, n in sorted(type_counts.items()):
    print(f"  {t:10s} : {n}")

Graph has 120 nodes and 630 edges.
  bulk       : 90
  surface    : 30


In [9]:
### Build reactants
from autokmc.reactants import build_reactant

c    = build_reactant("[C]", calculator=make_calc())
co   = build_reactant("[C-]#[O+]", calculator=make_calc())
o2   = build_reactant("O=O",       calculator=make_calc())
ch4  = build_reactant("C",         calculator=make_calc())
ch3  = build_reactant("[CH3]",     calculator=make_calc())
ch2  = build_reactant("[CH2]",     calculator=make_calc())
ch   = build_reactant("[CH]",      calculator=make_calc())
o    = build_reactant("[O]",       calculator=make_calc())

for r in [o, co, o2, ch4, ch3, ch2, ch]:
    print(f"\nReactant : {r.smiles}")
    print(f"  Formula  : {r.atoms.get_chemical_formula()}")
    print(f"  Atoms    : {len(r.atoms)}")
    print(f"  Nodes    : {r.graph.number_of_nodes()}")
    print(f"  Edges    : {r.graph.number_of_edges()}")
    for i, d in r.graph.nodes(data=True):
        pos = d['position']
        print(f"    node {i}  element={d['element']:2s}  type={d['type']}  r_cov={d['covalent_radius']:.3f} Å  pos=({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) Å")


Reactant : [O]
  Formula  : O
  Atoms    : 1
  Nodes    : 1
  Edges    : 0
    node 0  element=O   type=adsorbate  r_cov=0.660 Å  pos=(6.000, 6.000, 6.000) Å

Reactant : [C-]#[O+]
  Formula  : CO
  Atoms    : 2
  Nodes    : 2
  Edges    : 1
    node 0  element=C   type=adsorbate  r_cov=0.760 Å  pos=(7.125, 6.000, 6.000) Å
    node 1  element=O   type=adsorbate  r_cov=0.660 Å  pos=(5.993, 6.000, 6.000) Å

Reactant : O=O
  Formula  : O2
  Atoms    : 2
  Nodes    : 2
  Edges    : 1
    node 0  element=O   type=adsorbate  r_cov=0.660 Å  pos=(7.190, 6.000, 6.000) Å
    node 1  element=O   type=adsorbate  r_cov=0.660 Å  pos=(5.951, 6.000, 6.000) Å

Reactant : C
  Formula  : CH4
  Atoms    : 5
  Nodes    : 5
  Edges    : 4
    node 0  element=C   type=adsorbate  r_cov=0.760 Å  pos=(6.675, 6.836, 6.581) Å
    node 1  element=H   type=adsorbate  r_cov=0.310 Å  pos=(6.002, 7.688, 6.496) Å
    node 2  element=H   type=adsorbate  r_cov=0.310 Å  pos=(6.281, 6.002, 6.002) Å
    node 3  element=H   

In [10]:
### Site visualisation helper
# Each iso-class gets a distinct proxy element so they appear in different colours.
_ISO_PROXY = ["Au", "Pt", "Pd", "Ag", "Ir", "Rh", "Os", "Re"]

def _make_site_vis(slab, sites, reactant):
    """Return an Atoms object: slab + one representative per iso-class."""
    import copy as _copy
    vis = _copy.deepcopy(slab)
    seen_classes = set()
    for s in sites:
        if s.iso_class in seen_classes:
            continue
        seen_classes.add(s.iso_class)
        proxy = _ISO_PROXY[s.iso_class % len(_ISO_PROXY)]
        if len(reactant.atoms) == 1:
            vis.append(proxy)
            vis.positions[-1] = s.position
        else:
            ads_syms = reactant.atoms.get_chemical_symbols()
            for k, sym in enumerate(ads_syms):
                vis.append(proxy if k == 0 else sym)
                vis.positions[-1] = s.position[k]
    return vis

def _print_sites_single(label, sites):
    print(f"\n{label} sites summary")
    print(f"  Total sites  : {len(sites)}")
    for iso_cid, count in sorted(Counter(s.iso_class for s in sites).items()):
        rep = next(s for s in sites if s.iso_class == iso_cid)
        e_str = f"  E_ads={rep.energy:.4f} eV  converged={rep.converged}" if rep.energy is not None else ""
        print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites{e_str}  "
              f"pos = ({rep.position[0]:.3f}, {rep.position[1]:.3f}, {rep.position[2]:.3f}) Å")

def _print_sites_multi(label, sites):
    print(f"\n{label} sites summary")
    print(f"  Total sites  : {len(sites)}")
    for iso_cid, count in sorted(Counter(s.iso_class for s in sites).items()):
        rep = next(s for s in sites if s.iso_class == iso_cid)
        n_surf = len({sg for _, sg in rep.conn_global})
        e_str = f"  E_ads={rep.energy:.4f} eV  converged={rep.converged}" if rep.energy is not None else ""
        print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites  "
              f"{n_surf} surface atoms bonded{e_str}")

In [11]:
### Find adsorption sites — O (single atom, k_max=4)

sites_c, site_graph_c = find_adsorption_sites(
    graph, c,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_single("C", sites_c)

Finding sites for '[C]'  (single-atom path,  clash_factor=1.10 (= bond_factor))
  [single] surface representatives: 1 of 30 atoms  (wl_key=surf_wl_k1)
  [single] probe pts after filter: 308
  [single] unique connectivities: 34  (clash pruned=0)
  [single] after expansion: 270 instances
  [single pre-relax pruning] 2/5 classes pruned:
    iso-class  1: residual=12.843>2.0
    iso-class  4: residual=40.810>2.0

  EMT relaxation: 1 representative per iso-class (5 classes, 60 atoms frozen)
  iso-class  0  [FAIL]  converged=True  intended=1-fold  actual=0-fold
  iso-class  2  [OK  ]  converged=True  intended=3-fold  actual=3-fold  E_ads=-7.1029 eV
  iso-class  3  [OK  ]  converged=True  intended=3-fold  actual=3-fold  E_ads=-7.0493 eV
  Valid classes : 2 / 5

Site classification
  iso-class  2  hollow     30 sites  E_ads=-7.1029 eV  converged=True
  iso-class  3  hollow     30 sites  E_ads=-7.0493 eV  converged=True
  Total unique sites : 60
  Adjacency edges    : 360

C sites summary
  Tot

In [12]:
slab_c_vis = _make_site_vis(slab, sites_c, c)
print(f"C site visualisation: {len(slab_c_vis)} atoms "
      f"({len(slab)} slab + {len(slab_c_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_c}):
    rep = next(s for s in sites_c if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_c_vis)

C site visualisation: 122 atoms (120 slab + 2 representatives)
  iso-class 2  (hollow)  proxy=Pd  E_ads=-7.1029 eV
  iso-class 3  (hollow)  proxy=Ag  E_ads=-7.0493 eV


In [13]:
### Find adsorption sites — O (single atom, k_max=4)

sites_o, site_graph_o = find_adsorption_sites(
    graph, o,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_single("O", sites_o)

Finding sites for '[O]'  (single-atom path,  clash_factor=1.10 (= bond_factor))
  [single] surface representatives: 1 of 30 atoms  (wl_key=surf_wl_k1)
  [single] probe pts after filter: 295
  [single] unique connectivities: 22  (clash pruned=0)
  [single] after expansion: 180 instances
  [single pre-relax pruning] 1/4 classes pruned:
    iso-class  1: residual=14.214>2.0

  EMT relaxation: 1 representative per iso-class (4 classes, 60 atoms frozen)
  iso-class  0  [OK  ]  converged=True  intended=1-fold  actual=1-fold  E_ads=-5.8606 eV
  iso-class  2  [OK  ]  converged=True  intended=3-fold  actual=3-fold  E_ads=-8.0076 eV
  iso-class  3  [OK  ]  converged=True  intended=3-fold  actual=3-fold  E_ads=-7.9408 eV
  Valid classes : 3 / 4

Site classification
  iso-class  0  top        30 sites  E_ads=-5.8606 eV  converged=True
  iso-class  2  hollow     30 sites  E_ads=-8.0076 eV  converged=True
  iso-class  3  hollow     30 sites  E_ads=-7.9408 eV  converged=True
  Total unique sites : 90

In [14]:
slab_o_vis = _make_site_vis(slab, sites_o, o)
print(f"O site visualisation: {len(slab_o_vis)} atoms "
      f"({len(slab)} slab + {len(slab_o_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_o}):
    rep = next(s for s in sites_o if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_o_vis)

O site visualisation: 123 atoms (120 slab + 3 representatives)
  iso-class 0  (top)  proxy=Au  E_ads=-5.8606 eV
  iso-class 2  (hollow)  proxy=Pd  E_ads=-8.0076 eV
  iso-class 3  (hollow)  proxy=Ag  E_ads=-7.9408 eV


In [15]:
### Find adsorption sites — CO (multi-atom, k_max=3)

sites_co, site_graph_co = find_adsorption_sites(
    graph, co,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CO", sites_co)

Finding sites for '[C-]#[O+]'  (multi-atom path,  clash_factor=1.10 (= bond_factor))
  [multi] n_orientations auto=20 (R_max=0.566 Å, grid_spacing=0.400 Å)
  [multi] surface representatives: 1 of 30 atoms  (wl_key=surf_wl_k1, d_mol_max=0.566 Å)
  [multi] grid pts: 424 (after xy filter),  orientations: 20,  anchors: 2
  [multi] anchor candidates: [0, 1] (hull=[0, 1], WL-reps=[0, 1], final=2/2)
  [multi] unique connectivities: 744  (clash pruned=0)
  [multi] after expansion: 6750 instances
  [multi pre-relax pruning] 51/69 classes pruned:
    iso-class  6: spatial_dup_of_3 d=0.062Å
    iso-class  7: spatial_dup_of_4 d=0.062Å
    iso-class  8: stretch=1.347>1.3
    iso-class  9: stretch=1.347>1.3
    iso-class 10: residual=20.044>2.0
    iso-class 11: stretch=1.347>1.3
    iso-class 12: stretch=1.347>1.3
    iso-class 13: residual=13.763>2.0
    iso-class 14: residual=38.308>2.0
    iso-class 25: stretch=1.323>1.3
    iso-class 27: residual=43.229>2.0
    iso-class 28: residual=35.534>2.0

/Users/bunting4/local/lib/python3.13/site-packages/ase/io/extxyz.py:311: UserWarning: Skipping unhashable information frozen_indices
  warnings.warn('Skipping unhashable information '


  iso-class  0  [FAIL]  converged=True  surf-bonds intended=1  actual=1
  iso-class  1  [OK  ]  converged=True  surf-bonds intended=1  actual=1  E_ads=-17.1725 eV
  iso-class  2  [FAIL]  converged=True  surf-bonds intended=2  actual=3
  iso-class  3  [FAIL]  converged=True  surf-bonds intended=3  actual=3
  iso-class  4  [FAIL]  converged=True  surf-bonds intended=3  actual=1
  iso-class  5  [FAIL]  converged=True  surf-bonds intended=2  actual=3
  iso-class 15  [FAIL]  converged=True  surf-bonds intended=3  actual=3
  iso-class 16  [FAIL]  converged=True  surf-bonds intended=3  actual=3
  iso-class 17  [FAIL]  converged=True  surf-bonds intended=3  actual=0
  iso-class 18  [FAIL]  converged=True  surf-bonds intended=3  actual=0
  iso-class 19  [OK  ]  converged=True  surf-bonds intended=3  actual=3  E_ads=-17.4439 eV
  iso-class 20  [OK  ]  converged=True  surf-bonds intended=3  actual=3  E_ads=-17.3997 eV
  iso-class 21  [FAIL]  converged=True  surf-bonds intended=3  actual=3
  iso-c

In [16]:
slab_co_vis = _make_site_vis(slab, sites_co, co)
print(f"CO site visualisation: {len(slab_co_vis)} atoms "
      f"({len(slab)} slab + {len(slab_co_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_co}):
    rep = next(s for s in sites_co if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_co_vis)

CO site visualisation: 126 atoms (120 slab + 6 representatives)
  iso-class 1  (top)  proxy=Pt  E_ads=-17.1725 eV
  iso-class 19  (hollow)  proxy=Ag  E_ads=-17.4439 eV
  iso-class 20  (hollow)  proxy=Ir  E_ads=-17.3997 eV


In [17]:
### Find adsorption sites — O2 (multi-atom, k_max=4)

sites_o2, site_graph_o2 = find_adsorption_sites(
    graph, o2,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("O2", sites_o2)

Finding sites for 'O=O'  (multi-atom path,  clash_factor=1.10 (= bond_factor))
  [multi] n_orientations auto=24 (R_max=0.619 Å, grid_spacing=0.400 Å)
  [multi] surface representatives: 1 of 30 atoms  (wl_key=surf_wl_k1, d_mol_max=0.619 Å)
  [multi] grid pts: 328 (after xy filter),  orientations: 24,  anchors: 2
  [multi] anchor candidates: [0] (hull=[0, 1], WL-reps=[0], final=1/2)
  [multi] unique connectivities: 279  (clash pruned=0)
  [multi] after expansion: 5160 instances
  [multi pre-relax pruning] 19/34 classes pruned:
    iso-class  6: spatial_dup_of_3 d=0.089Å
    iso-class  7: spatial_dup_of_4 d=0.089Å
    iso-class 12: residual=15.003>2.0
    iso-class 13: residual=39.371>2.0
    iso-class 14: stretch=1.500>1.3
    iso-class 19: spatial_dup_of_16 d=0.113Å
    iso-class 21: stretch=1.396>1.3
    iso-class 22: stretch=1.604>1.3
    iso-class 23: stretch=1.990>1.3
    iso-class 24: stretch=1.514>1.3
    iso-class 25: stretch=1.514>1.3
    iso-class 26: stretch=1.514>1.3
    iso-

In [18]:
slab_o2_vis = _make_site_vis(slab, sites_o2, o2)
print(f"O2 site visualisation: {len(slab_o2_vis)} atoms "
      f"({len(slab)} slab + {len(slab_o2_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_o2}):
    rep = next(s for s in sites_o2 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_o2_vis)

O2 site visualisation: 130 atoms (120 slab + 10 representatives)
  iso-class 1  (top)  proxy=Pt  E_ads=-11.1387 eV
  iso-class 8  (hollow)  proxy=Au  E_ads=-13.1050 eV
  iso-class 9  (hollow)  proxy=Pt  E_ads=-13.1119 eV
  iso-class 17  (hollow)  proxy=Pt  E_ads=-12.0864 eV
  iso-class 18  (hollow)  proxy=Pd  E_ads=-12.0321 eV


In [19]:
### Find adsorption sites — CH (multi-atom, k_max=4)

sites_ch, site_graph_ch = find_adsorption_sites(
    graph, ch,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH", sites_ch)

Finding sites for '[CH]'  (multi-atom path,  clash_factor=1.10 (= bond_factor))
  [multi] n_orientations auto=20 (R_max=0.566 Å, grid_spacing=0.400 Å)
  [multi] surface representatives: 1 of 30 atoms  (wl_key=surf_wl_k1, d_mol_max=0.566 Å)
  [multi] grid pts: 574 (after xy filter),  orientations: 20,  anchors: 2
  [multi] anchor candidates: [0, 1] (hull=[0, 1], WL-reps=[0, 1], final=2/2)
  [multi] unique connectivities: 593  (clash pruned=0)
  [multi] after expansion: 5580 instances
  [multi pre-relax pruning] 44/54 classes pruned:
    iso-class  2: stretch=1.403>1.3
    iso-class  3: stretch=1.403>1.3
    iso-class  4: stretch=1.475>1.3
    iso-class  5: stretch=1.475>1.3
    iso-class  6: residual=21.296>2.0
    iso-class  7: residual=13.706>2.0
    iso-class  8: residual=41.574>2.0
    iso-class  9: stretch=1.347>1.3
    iso-class 10: stretch=1.347>1.3
    iso-class 11: stretch=1.347>1.3
    iso-class 12: stretch=1.347>1.3
    iso-class 17: spatial_dup_of_13 d=0.060Å
    iso-class 1

In [20]:
slab_ch_vis = _make_site_vis(slab, sites_ch, ch)
print(f"CH site visualisation: {len(slab_ch_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch}):
    rep = next(s for s in sites_ch if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch_vis)

CH site visualisation: 128 atoms (120 slab + 8 representatives)
  iso-class 0  (top)  proxy=Au  E_ads=-10.1731 eV
  iso-class 15  (hollow)  proxy=Re  E_ads=-12.0583 eV
  iso-class 16  (hollow)  proxy=Au  E_ads=-11.9926 eV
  iso-class 18  (hollow)  proxy=Pd  E_ads=-9.6790 eV


In [21]:
### Find adsorption sites — CH2 (multi-atom, k_max=4)

sites_ch2, site_graph_ch2 = find_adsorption_sites(
    graph, ch2,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH2", sites_ch2)

Finding sites for '[CH2]'  (multi-atom path,  clash_factor=1.10 (= bond_factor))
  [multi] n_orientations auto=69 (R_max=1.053 Å, grid_spacing=0.400 Å)
  [multi] surface representatives: 1 of 30 atoms  (wl_key=surf_wl_k1, d_mol_max=1.053 Å)
  [multi] grid pts: 574 (after xy filter),  orientations: 69,  anchors: 3
  [multi] anchor candidates: [0, 1] (hull=[0, 1, 2], WL-reps=[0, 1], final=2/3)
  [multi] unique connectivities: 2410  (clash pruned=0)
  [multi] after expansion: 41040 instances
  [multi pre-relax pruning] 174/182 classes pruned:
    iso-class  1: spatial_dup_of_0 d=0.000Å
    iso-class  2: stretch=1.452>1.3
    iso-class  3: stretch=1.451>1.3
    iso-class  4: stretch=1.516>1.3
    iso-class  5: stretch=1.516>1.3
    iso-class  6: stretch=1.517>1.3
    iso-class  7: stretch=1.517>1.3
    iso-class  8: residual=21.810>2.0
    iso-class  9: residual=41.594>2.0
    iso-class 10: stretch=1.366>1.3
    iso-class 11: stretch=1.366>1.3
    iso-class 12: residual=30.495>2.0
    iso-

In [22]:
slab_ch2_vis = _make_site_vis(slab, sites_ch2, ch2)
print(f"CH2 site visualisation: {len(slab_ch2_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch2_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch2}):
    rep = next(s for s in sites_ch2 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch2_vis)

CH2 site visualisation: 123 atoms (120 slab + 3 representatives)
  iso-class 0  (top)  proxy=Au  E_ads=-15.1606 eV


In [ ]:
### Find adsorption sites — CH3 (multi-atom, k_max=4)

sites_ch3, site_graph_ch3 = find_adsorption_sites(
    graph, ch3,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH3", sites_ch3)

Finding sites for '[CH3]'  (multi-atom path,  clash_factor=1.10 (= bond_factor))
  [multi] n_orientations auto=73 (R_max=1.083 Å, grid_spacing=0.400 Å)
  [multi] surface representatives: 1 of 30 atoms  (wl_key=surf_wl_k1, d_mol_max=1.083 Å)
  [multi] grid pts: 574 (after xy filter),  orientations: 73,  anchors: 4
  [multi] anchor candidates: [0, 1] (hull=[0, 1, 2, 3], WL-reps=[0, 1], final=2/4)
  [multi] unique connectivities: 5606  (clash pruned=0)


In [ ]:
slab_ch3_vis = _make_site_vis(slab, sites_ch3, ch3)
print(f"CH3 site visualisation: {len(slab_ch3_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch3_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch3}):
    rep = next(s for s in sites_ch3 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch3_vis)

In [ ]:
### Find adsorption sites — CH4 (multi-atom, k_max=4)

sites_ch4, site_graph_ch4 = find_adsorption_sites(
    graph, ch4,
    calculator=make_calc(),
    slab=slab,
    verbose=True,
)
_print_sites_multi("CH4", sites_ch4)

In [ ]:
slab_ch4_vis = _make_site_vis(slab, sites_ch4, ch4)
print(f"CH4 site visualisation: {len(slab_ch4_vis)} atoms "
      f"({len(slab)} slab + {len(slab_ch4_vis) - len(slab)} representatives)")
for iso_cid in sorted({s.iso_class for s in sites_ch4}):
    rep = next(s for s in sites_ch4 if s.iso_class == iso_cid)
    proxy = _ISO_PROXY[iso_cid % len(_ISO_PROXY)]
    e_str = f"  E_ads={rep.energy:.4f} eV" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid}  ({rep.site_type})  proxy={proxy}{e_str}")
view_x3d(slab_ch4_vis)